# Multi-Head Attention — Solution

Reference implementation for `03_multi_head_attention_exercise.ipynb`.

## Setup

In [1]:
import torch
import torch.nn as nn

## Solution 1 — `MultiHeadAttention`

In [19]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, n_heads, causal=False):
        super().__init__()
        self.d_model = d_model
        self.n_heads = n_heads
        self.d_head = d_model//n_heads
        self.causal = causal

        self.qkv_projection = nn.Linear(d_model,d_model*3)
        self.out_projection = nn.Linear(d_model,d_model)

    def forward(self, x):
        # x -> bs,seq_len,d_model
        bs,seq_len,_ = x.shape 

        Q,K,V = torch.chunk(self.qkv_projection(x),3,dim=-1) # bs,seq_len,d_model

        Q = Q.view(bs,seq_len,self.n_heads,self.d_head).permute(0,2,1,3) #bs,n_heads,seq_len,d_head
        K = K.view(bs,seq_len,self.n_heads,self.d_head).permute(0,2,1,3) #bs,n_heads,seq_len,d_head
        V = V.view(bs,seq_len,self.n_heads,self.d_head).permute(0,2,1,3) #bs,n_heads,seq_len,d_head

        scores = Q@K.transpose(-2,-1) / self.d_head**0.5

        if self.causal:
            mask = torch.tril(torch.ones(seq_len,seq_len,dtype=torch.bool))
            scores = scores.masked_fill(~mask,float("-inf"))

        weights = torch.softmax(scores,dim=-1)
        out = weights@V

        out = out.permute(0,2,1,3).contiguous().view(bs,seq_len,self.d_model)
        out = self.out_projection(out)

        return out

**Why one big matrix + reshape, not `n_heads` small matrices?** _Explain: GPUs love one large matmul over many small ones; the math is identical because `(d_model, d_model)` reshaped to `(n_heads, d_head, d_model)` and split is exactly `n_heads` separate `(d_model, d_head)` matrices stacked. This view is essential for understanding MQA/GQA in Part 2 — the only difference there is splitting K/V differently than Q._

## Solution 2 — Verify causal MultiHeadAttention

In [20]:
mha = MultiHeadAttention(d_model=64, n_heads=8, causal=True)
x_a = torch.randn(1, 5, 64)
x_b = x_a.clone()
x_b[0, 4] = torch.randn(64)        # change last token only

out_a = mha(x_a)
out_b = mha(x_b)

In [22]:
# Earlier positions: should be IDENTICAL — they can't see position 4
for i in range(4):
    same = torch.allclose(out_a[0, i], out_b[0, i], atol=1e-5)
    print(f"position {i}: {'✓ unchanged' if same else '✗ LEAKED'}")

# Last position: SHOULD differ (it sees its own change)
last_differs = not torch.allclose(out_a[0, 4], out_b[0, 4])
print(f"position 4: {'✓ changed (as expected)' if last_differs else '✗ unchanged?!'}")


print("\n✓ Causal MultiHeadAttention isolates each position to its past")

position 0: ✓ unchanged
position 1: ✓ unchanged
position 2: ✓ unchanged
position 3: ✓ unchanged
position 4: ✓ changed (as expected)

✓ Causal MultiHeadAttention isolates each position to its past


## Run the tests

In [23]:
from tests import run_multi_head_tests
run_multi_head_tests(MultiHeadAttention)

Running MultiHeadAttention...
  ✓ Output shape correct
  ✓ Parameter count in expected range
  ✓ Gradients flow
  ✓ Causal mode: future doesn't leak
  ✗ Asserts d_model divisible by n_heads — d_model not divisible by n_heads should raise

  4/5 checks passed — fix the ✗ above



False